# GPT-124M Architecture

A from-scratch implementation of the GPT-2 (124M) transformer, assembled from:
**token + positional embeddings → N transformer blocks → final LayerNorm → output head.**

Each transformer block uses *pre-LayerNorm*, multi-head causal self-attention,
a GELU feed-forward network, and residual (shortcut) connections.

In [ ]:
import math
import torch
import torch.nn as nn

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Max context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of transformer blocks
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False,      # Bias for the Q/K/V projections
}

## Layer Normalization

Layer normalization normalizes *across the embedding features* of each token,
then applies a learnable scale and shift. In a modern GPT block it is applied
**before** each sub-layer (pre-norm):

```
norm(x) = (x - mean) / sqrt(var + eps) * scale + shift
```

Within a transformer block normalization happens twice — once before attention
and once before the feed-forward network — and a final norm is applied after the
last block.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # Small constant to avoid division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

## GELU Activation

GELU (Gaussian Error Linear Unit) is the activation used inside the feed-forward
network. We use the tanh approximation from the GPT-2 paper.

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))

## Feed-Forward Network

A position-wise MLP that expands the embedding dimension by 4x, applies GELU,
and projects back down.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

## Multi-Head Causal Self-Attention

Implemented as a single class with weight splits: the Q/K/V projections produce a
combined `emb_dim` output that is reshaped into `n_heads` heads of size
`head_dim = emb_dim / n_heads`. A causal mask prevents each token from attending
to future tokens.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["emb_dim"] % cfg["n_heads"] == 0, "emb_dim must be divisible by n_heads"

        self.d_out = cfg["emb_dim"]
        self.n_heads = cfg["n_heads"]
        self.head_dim = self.d_out // self.n_heads

        self.W_q = nn.Linear(cfg["emb_dim"], self.d_out, bias=cfg["qkv_bias"])
        self.W_k = nn.Linear(cfg["emb_dim"], self.d_out, bias=cfg["qkv_bias"])
        self.W_v = nn.Linear(cfg["emb_dim"], self.d_out, bias=cfg["qkv_bias"])
        self.out_proj = nn.Linear(self.d_out, self.d_out)
        self.dropout = nn.Dropout(cfg["drop_rate"])

        # Causal mask: upper-triangular (excluding diagonal) is masked out
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(cfg["context_length"], cfg["context_length"]),
                diagonal=1,
            ).bool(),
        )

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape

        queries = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        # (batch, num_tokens, d_out) -> (batch, n_heads, num_tokens, head_dim)
        def split_heads(t):
            return t.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)

        queries = split_heads(queries)
        keys = split_heads(keys)
        values = split_heads(values)

        # Scaled dot-product attention scores per head
        attn_scores = queries @ keys.transpose(2, 3)

        # Apply the causal mask
        mask = self.mask[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask, float("-inf"))

        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values, then merge heads back together
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(batch_size, num_tokens, self.d_out)

        return self.out_proj(context_vec)

## Transformer Block

Pre-norm architecture with residual connections around both sub-layers:

```
x = x + Dropout(Attention(LayerNorm(x)))
x = x + Dropout(FeedForward(LayerNorm(x)))
```

The residual (shortcut) connections add the *unmodified* input back, which keeps
gradients flowing through deep stacks of blocks.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg)
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Attention sub-layer with residual connection
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Feed-forward sub-layer with residual connection
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

## GPT Model

Ties the components together: embeddings, a stack of transformer blocks, a final
LayerNorm, and a linear head projecting back to the vocabulary.

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

## Greedy Text Generation

Autoregressively appends one token at a time by taking the argmax of the
logits for the last position.

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        # Crop the context to the supported length
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus on the last time step, then pick the most likely next token
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

## Example: Run the Untrained Model

The model is randomly initialized, so the generated text is gibberish — this just
verifies the architecture runs end-to-end.

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded:", encoded)
print("encoded_tensor.shape:", encoded_tensor.shape)

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()  # disable dropout for deterministic generation

n_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", f"{n_params:,}")

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)
print("Output shape:", out.shape)
print("Output text:", tokenizer.decode(out.squeeze(0).tolist()))